# Agentic Patterns with SingleStore

In [6]:
!pip cache purge --quiet

In [7]:
!pip install openai==2.2.0 --quiet

In [8]:
import json
import logging
import os
import random

from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from datetime import datetime, timedelta
from openai import OpenAI
from singlestoredb.management import get_secret
from typing import Dict, List, Optional, Tuple

In [9]:
# Suppress httpx logging

logging.getLogger("httpx").setLevel(logging.WARNING)

In [10]:
# Global Configuration

LLM_MODEL = "gpt-4.1-mini"
TEMPERATURE = 0.0
MAX_TOKENS = 300
SEED = 42

In [11]:
# Initialize clients

os.environ["OPENAI_API_KEY"] = get_secret("OPENAI_API_KEY")
openai_client = OpenAI()

In [12]:
# Data Models

@dataclass
class AgentDecision:
    """Structured decision output from each agent."""
    agent_id: str
    recommendation: str
    confidence: float
    reasoning: str
    metadata: Dict = field(default_factory = dict)

@dataclass
class TradingContext:
    """Shared context passed through the agent chain."""
    symbol: str
    price_data: str = ""
    sentiment_data: str = ""
    position_data: str = ""
    recent_decisions: str = ""
    portfolio_data: str = ""
    agent_decisions: List[AgentDecision] = field(default_factory = list)

In [13]:
# Data Access Layer

def query_db(sql: str, params: dict = None) -> List[Dict]:
    """Execute query and return results."""
    with db_connection.connect() as conn:
        if params is None:
            params = {}
        result = conn.execute(text(sql), params)
        return [dict(row._mapping) for row in result]

def execute_db(sql: str, params: dict = None):
    """Execute insert/update statement."""
    with db_connection.connect() as conn:
        if params is None:
            params = {}
        conn.execute(text(sql), params)
        conn.commit()

def get_recent_ticks(symbol: str, hours: int = 24) -> str:
    """Fetch recent price data."""
    ticks = query_db(
        """SELECT close, high, low, volume
           FROM tick
           WHERE symbol = :symbol AND ts >= DATE_SUB(NOW(), INTERVAL :hours HOUR)
           ORDER BY ts DESC LIMIT 100""",
        {"symbol": symbol, "hours": hours}
    )
    if not ticks:
        return f"No data for {symbol}"

    avg_vol = sum(t["volume"] for t in ticks) // len(ticks)
    high = max(t["high"] for t in ticks)
    low = min(t["low"] for t in ticks)
    current = ticks[0]["close"]
    oldest = ticks[-1]["close"]
    pct_change = (current - oldest) / oldest * 100

    return f"Price: ${current:.2f} | {hours}h Change: {pct_change:+.1f}% | {hours}h High: ${high:.2f} | Low: ${low:.2f} | Avg Vol: {avg_vol:,}"

def get_recent_sentiment(symbol: str, days: int = 7) -> str:
    """Fetch sentiment analysis."""
    sentiments = query_db(
        """SELECT headline, compound
           FROM stock_sentiment
           WHERE symbol = :symbol AND ts >= DATE_SUB(NOW(), INTERVAL :days DAY)
           ORDER BY ts DESC LIMIT 20""",
        {"symbol": symbol, "days": days}
    )
    if not sentiments:
        return f"No sentiment data for {symbol}"

    avg_compound = sum(s["compound"] for s in sentiments) / len(sentiments)
    pos_count = sum(1 for s in sentiments if s["compound"] > 0.05)
    neg_count = sum(1 for s in sentiments if s["compound"] < -0.05)

    return f"Sentiment Score: {avg_compound:.2f} | Positive: {pos_count} | Negative: {neg_count}"

def get_portfolio_position(symbol: str) -> str:
    """Get current holdings."""
    pos = query_db(
        "SELECT shares_held, purchase_price FROM portfolio WHERE symbol = :symbol",
        {"symbol": symbol}
    )
    if not pos:
        return f"No position in {symbol}"

    return f"Holding: {pos[0]['shares_held']} shares @ ${pos[0]['purchase_price']:.2f}"

def get_recent_decisions(symbol: str, hours: int = 24) -> str:
    """Fetch recent agent decisions."""
    decisions = query_db(
        """SELECT agent_id, action, confidence
           FROM agent_decisions
           WHERE symbol = :symbol AND decision_timestamp >= DATE_SUB(NOW(), INTERVAL :hours HOUR)
           ORDER BY decision_timestamp DESC LIMIT 5""",
        {"symbol": symbol, "hours": hours}
    )
    if not decisions:
        return f"No recent decisions for {symbol}"

    return " | ".join(f"{d['agent_id']}: {d['action']} ({d['confidence']*100:.0f}%)" for d in decisions)

def get_all_positions() -> str:
    """Get full portfolio."""
    positions = query_db(
        "SELECT symbol, shares_held, purchase_price FROM portfolio ORDER BY symbol"
    )
    if not positions:
        return "Portfolio is empty"

    return " | ".join(f"{p['symbol']}: {p['shares_held']} @ ${p['purchase_price']:.2f}" for p in positions)

def log_decision(decision: AgentDecision, symbol: str):
    """Log agent decision to database."""
    execute_db(
        """INSERT INTO agent_decisions
           (agent_id, decision_timestamp, symbol, action, confidence, reasoning, data_sources)
           VALUES (:agent_id, NOW(), :symbol, :action, :confidence, :reasoning, :data_sources)""",
        {
            "agent_id": decision.agent_id,
            "symbol": symbol,
            "action": decision.recommendation,
            "confidence": decision.confidence,
            "reasoning": decision.reasoning,
            "data_sources": "market_data"
        }
    )

def execute_trade(symbol: str, action: str, shares: int):
    """Execute a trade."""
    if action == "BUY":
        price = query_db(
            "SELECT close FROM tick WHERE symbol = :symbol ORDER BY ts DESC LIMIT 1",
            {"symbol": symbol}
        )
        purchase_price = price[0]["close"] if price else 100.0

        # Check if position exists
        existing = query_db(
            "SELECT shares_held FROM portfolio WHERE symbol = :symbol",
            {"symbol": symbol}
        )

        if existing:
            # Update existing position
            execute_db(
                "UPDATE portfolio SET shares_held = shares_held + :shares WHERE symbol = :symbol",
                {"shares": shares, "symbol": symbol}
            )
        else:
            # Create new position
            execute_db(
                """INSERT INTO portfolio (symbol, shares_held, purchase_date, purchase_price)
                   VALUES (:symbol, :shares, CURDATE(), :price)""",
                {"symbol": symbol, "shares": shares, "price": purchase_price}
            )
        print(f"*** TRADE EXECUTED: BUY {shares} shares of {symbol} @ ${purchase_price:.2f}")
    elif action == "SELL":
        # Check if enough shares to sell
        position = query_db(
            "SELECT shares_held FROM portfolio WHERE symbol = :symbol",
            {"symbol": symbol}
        )

        if not position:
            print(f"!!! TRADE FAILED: No position in {symbol} to sell")
            return

        current_shares = position[0]['shares_held']
        if current_shares < shares:
            print(f"!!! TRADE FAILED: Cannot sell {shares} shares, only have {current_shares}")
            return

        execute_db(
            "UPDATE portfolio SET shares_held = shares_held - :shares WHERE symbol = :symbol",
            {"shares": shares, "symbol": symbol}
        )
        print(f"*** TRADE EXECUTED: SELL {shares} shares of {symbol}")

In [14]:
# Chain of Responsibility Pattern

class TradingAgent(ABC):
    """Abstract base class for agents in the chain."""

    def __init__(self, agent_id: str, next_agent: Optional["TradingAgent"] = None):
        self.agent_id = agent_id
        self.next_agent = next_agent

    def set_next(self, agent: "TradingAgent") -> "TradingAgent":
        """Set the next agent in the chain."""
        self.next_agent = agent
        return agent

    def process(self, context: TradingContext) -> TradingContext:
        """Process the request and pass to next agent."""
        # Load data if needed
        context = self.load_data(context)

        # Make decision
        decision = self.make_decision(context)
        context.agent_decisions.append(decision)

        # Log decision
        log_decision(decision, context.symbol)

        # Display
        self.display_decision(decision)

        # Pass to next agent
        if self.next_agent:
            return self.next_agent.process(context)
        return context

    @abstractmethod
    def load_data(self, context: TradingContext) -> TradingContext:
        """Load necessary data for this agent."""
        pass

    @abstractmethod
    def make_decision(self, context: TradingContext) -> AgentDecision:
        """Make a decision based on context."""
        pass

    def display_decision(self, decision: AgentDecision):
        """Display the agent's decision."""
        print(f"\n{'-' * 60}")
        print(f"[AGENT] {self.agent_id.upper()}")
        print(f"{'-' * 60}")
        print(f"Recommendation: {decision.recommendation}")
        print(f"Confidence: {decision.confidence * 100:.0f}%")
        print(f"Reasoning: {decision.reasoning}")
        if decision.metadata:
            print(f"Metadata: {decision.metadata}")

    def call_llm(self, messages: List[Dict], temp: float = None) -> str:
        """Call OpenAI API."""
        if temp is None:
            temp = TEMPERATURE
        response = openai_client.chat.completions.create(
            model = LLM_MODEL,
            messages = messages,
            temperature = temp,
            max_tokens = MAX_TOKENS
        )
        return response.choices[0].message.content

In [15]:
# Concrete Agent Implementation

class DataAnalystAgent(TradingAgent):
    """Stage 1: Analyzes market data and sentiment."""

    def load_data(self, context: TradingContext) -> TradingContext:
        context.price_data = get_recent_ticks(context.symbol)
        context.sentiment_data = get_recent_sentiment(context.symbol)
        context.position_data = get_portfolio_position(context.symbol)
        return context

    def make_decision(self, context: TradingContext) -> AgentDecision:
        prompt = f"""You are a Data Analyst. Analyze this market data for {context.symbol}.

Price Data: {context.price_data}
Sentiment: {context.sentiment_data}
Current Position: {context.position_data}

Provide a trading recommendation (BUY, SELL, or HOLD) based on technical and sentiment analysis.

Respond ONLY with valid JSON in this exact format:
{{"recommendation": "BUY", "confidence": 0.75, "reasoning": "Strong positive sentiment with price momentum"}}"""

        response = self.call_llm([{"role": "user", "content": prompt}])

        try:
            # Strip markdown code blocks if present
            cleaned = response.strip()
            if cleaned.startswith("```"):
                cleaned = cleaned.split("```")[1]
                if cleaned.startswith("json"):
                    cleaned = cleaned[4:]
            cleaned = cleaned.strip()

            data = json.loads(cleaned)
            return AgentDecision(
                agent_id = self.agent_id,
                recommendation = data["recommendation"],
                confidence = float(data["confidence"]),
                reasoning = data["reasoning"],
                metadata = {"approved": True, "fallback": False}
            )
        except (json.JSONDecodeError, KeyError) as e:
            print(f"!!! Parse error: {e}. Using fallback.")
            return AgentDecision(
                agent_id = self.agent_id,
                recommendation = "HOLD",
                confidence = 0.5,
                reasoning = "Unable to parse LLM response",
                metadata = {"fallback": True, "error": str(e)}
            )

class RiskManagerAgent(TradingAgent):
    """Stage 2: Reviews and validates recommendations."""

    def load_data(self, context: TradingContext) -> TradingContext:
        context.recent_decisions = get_recent_decisions(context.symbol)
        return context

    def make_decision(self, context: TradingContext) -> AgentDecision:
        analyst_decision = context.agent_decisions[-1]

        prompt = f"""You are a Risk Manager. Review this trading recommendation for {context.symbol}.

Analyst Recommendation: {analyst_decision.recommendation}
Analyst Confidence: {analyst_decision.confidence * 100:.0f}%
Analyst Reasoning: {analyst_decision.reasoning}

Recent Trading Activity: {context.recent_decisions}

Assess risk factors:
- Is confidence level adequate (>60%)?
- Are we trading too frequently?
- Is the recommendation reasonable given recent activity?

Approve, modify, or reject the recommendation.

Respond ONLY with valid JSON in this exact format:
{{"recommendation": "BUY", "confidence": 0.80, "reasoning": "Approved with high confidence", "approved": true}}"""

        response = self.call_llm([{"role": "user", "content": prompt}])

        try:
            cleaned = response.strip()
            if cleaned.startswith("```"):
                cleaned = cleaned.split("```")[1]
                if cleaned.startswith("json"):
                    cleaned = cleaned[4:]
            cleaned = cleaned.strip()

            data = json.loads(cleaned)
            return AgentDecision(
                agent_id = self.agent_id,
                recommendation = data["recommendation"],
                confidence = float(data["confidence"]),
                reasoning = data["reasoning"],
                metadata = {
                    "approved": data.get("approved", True),
                    "fallback": False
                }
            )
        except (json.JSONDecodeError, KeyError) as e:
            print(f"!!! Parse error: {e}. Defaulting to HOLD.")
            return AgentDecision(
                agent_id = self.agent_id,
                recommendation = "HOLD",
                confidence = 0.5,
                reasoning = "Risk assessment failed - defaulting to HOLD",
                metadata = {"approved": False, "fallback": True, "error": str(e)}
            )

class PortfolioOptimizerAgent(TradingAgent):
    """Stage 3: Optimizes for portfolio balance and diversification."""

    def load_data(self, context: TradingContext) -> TradingContext:
        context.portfolio_data = get_all_positions()
        return context

    def make_decision(self, context: TradingContext) -> AgentDecision:
        risk_decision = context.agent_decisions[-1]

        prompt = f"""You are a Portfolio Optimizer. Finalize this trade decision for {context.symbol}.

Risk-Approved Recommendation: {risk_decision.recommendation}
Risk Manager Confidence: {risk_decision.confidence * 100:.0f}%
Risk Manager Notes: {risk_decision.reasoning}

Full Portfolio: {context.portfolio_data}

Consider:
- Does this improve diversification?
- What is the optimal trade size (10-100 shares)?
- Should we proceed with this trade?

Provide final recommendation and trade size.

Respond ONLY with valid JSON in this exact format:
{{"recommendation": "BUY", "confidence": 0.85, "reasoning": "Improves diversification", "trade_size": 25}}"""

        response = self.call_llm([{"role": "user", "content": prompt}])

        try:
            cleaned = response.strip()
            if cleaned.startswith("```"):
                cleaned = cleaned.split("```")[1]
                if cleaned.startswith("json"):
                    cleaned = cleaned[4:]
            cleaned = cleaned.strip()

            data = json.loads(cleaned)
            return AgentDecision(
                agent_id = self.agent_id,
                recommendation = data["recommendation"],
                confidence = float(data["confidence"]),
                reasoning = data["reasoning"],
                metadata = {
                    "trade_size": data.get("trade_size", 10),
                    "approved": True,
                    "fallback": False
                }
            )
        except (json.JSONDecodeError, KeyError) as e:
            print(f"!!! Parse error: {e}. Defaulting to HOLD.")
            return AgentDecision(
                agent_id = self.agent_id,
                recommendation = "HOLD",
                confidence = 0.5,
                reasoning = "Optimization failed - defaulting to HOLD",
                metadata = {
                    "trade_size": 0,
                    "approved": False,
                    "fallback": True,
                    "error": str(e)
                }
            )

In [16]:
# Orchestration and Execution

def run_trading_pipeline(symbol: str, execute: bool = False) -> TradingContext:
    """Run the complete multi-agent trading pipeline."""

    print(f"\n{'=' * 60}")
    print(f"[SYSTEM] MULTI-AGENT TRADING SYSTEM: {symbol}")
    print(f"{'=' * 60}")

    # Build the chain of responsibility
    analyst = DataAnalystAgent("data_analyst")
    risk_mgr = RiskManagerAgent("risk_manager")
    optimizer = PortfolioOptimizerAgent("portfolio_optimizer")

    analyst.set_next(risk_mgr).set_next(optimizer)

    # Initialize context
    context = TradingContext(symbol = symbol)

    # Run through the chain
    context = analyst.process(context)

    # Execute trade if recommended
    final_decision = context.agent_decisions[-1]

    print(f"\n{'=' * 60}")
    print(f"[FINAL] DECISION")
    print(f"{'=' * 60}")
    print(f"Action: {final_decision.recommendation}")
    print(f"Trade Size: {final_decision.metadata.get('trade_size', 0)} shares")
    print(f"Confidence: {final_decision.confidence * 100:.0f}%")
    print(f"Approved: {final_decision.metadata.get('approved', False)}")
    print(f"Fallback: {final_decision.metadata.get('fallback', False)}")

    if execute and final_decision.recommendation in ["BUY", "SELL"]:
        trade_size = final_decision.metadata.get("trade_size", 10)
        execute_trade(symbol, final_decision.recommendation, trade_size)
    elif not execute:
        print("\n[INFO] Run with execute = True to execute trades")

    return context

<div class="alert alert-block alert-warning">
    <b class="fa fa-solid fa-exclamation-circle"></b>
    <div>
        <p><b>Action Required</b></p>
        <p>Select the database from the drop-down menu at the top of this notebook. It updates the <b>connection_url</b> which is used by SQLAlchemy to make connections to the selected database.</p>
    </div>
</div>

In [17]:
from sqlalchemy import *

db_connection = create_engine(connection_url)

In [18]:
# Setup and Schema Creation

def setup_database_schema():
    '''Create all required tables for the trading system.'''

    # Drop existing tables for clean slate
    print("Dropping existing tables...")
    execute_db("DROP TABLE IF EXISTS agent_decisions")
    execute_db("DROP TABLE IF EXISTS portfolio")
    execute_db("DROP TABLE IF EXISTS stock_sentiment")
    execute_db("DROP TABLE IF EXISTS tick")
    print("Existing tables dropped")

    # Market tick data (price history)
    execute_db('''
        CREATE TABLE IF NOT EXISTS tick (
            symbol VARCHAR(10),
            ts     DATETIME SERIES TIMESTAMP,
            open   NUMERIC(18, 2),
            high   NUMERIC(18, 2),
            low    NUMERIC(18, 2),
            close  NUMERIC(18, 2),
            volume INT,
            PRIMARY KEY (symbol, ts)
        )
    ''')

    # Sentiment data (news/social sentiment)
    execute_db('''
        CREATE TABLE IF NOT EXISTS stock_sentiment (
            headline  VARCHAR(250),
            compound  FLOAT,
            positive  FLOAT,
            negative  FLOAT,
            neutral   FLOAT,
            url       TEXT,
            publisher VARCHAR(30),
            ts        DATETIME,
            symbol    VARCHAR(10),
            PRIMARY KEY (symbol, ts, headline(100))
        )
    ''')

    # Portfolio holdings
    execute_db('''
        CREATE TABLE IF NOT EXISTS portfolio (
            symbol         VARCHAR(10) PRIMARY KEY,
            shares_held    INT DEFAULT 0,
            purchase_date  DATE,
            purchase_price NUMERIC(18, 2)
        )
    ''')

    # Agent decisions (audit trail)
    execute_db('''
        CREATE TABLE IF NOT EXISTS agent_decisions (
            agent_id           VARCHAR(50),
            decision_timestamp DATETIME,
            symbol             VARCHAR(10),
            action             VARCHAR(10),
            confidence         DECIMAL(5, 4),
            reasoning          TEXT,
            data_sources       VARCHAR(100),
            PRIMARY KEY (symbol, decision_timestamp, agent_id)
        )
    ''')
    
    print("Database schema created successfully!")

setup_database_schema()

Dropping existing tables...
Existing tables dropped
Database schema created successfully!


In [19]:
# Load sample data for testing

def load_sample_data():
    '''Load sample data with a distinct story per symbol, so agents see real signal instead of noise.

    BBRQ-FX: clean breakout - steady uptrend, building volume, strongly positive sentiment
    BJBY-FX: selloff with a late contrarian rumor - downtrend, panic volume, one bullish outlier headline
    YWMG-FX: choppy/range-bound - deliberately balanced price and sentiment, plus a late volatility spike
    '''

    random.seed(SEED)

    now = datetime.now()
    symbols = ["BBRQ-FX", "BJBY-FX", "YWMG-FX"]
    base_prices = {"BBRQ-FX": 180.0, "BJBY-FX": 140.0, "YWMG-FX": 380.0}

    # Sample tick data (last 48 hours)
    print("Loading sample tick data...")
    for symbol in symbols:
        base = base_prices[symbol]
        for i in range(100):
            ts = now - timedelta(hours = 48 - i * 0.5)
            progress = i / 99

            if symbol == "BBRQ-FX":
                # Steady uptrend that accelerates into a breakout, with volume building alongside it
                drift = progress * 30
                price = base + drift + random.uniform(-1.5, 1.5)
                volume = int(1_500_000 + progress * 4_000_000 + random.randint(-200000, 200000))
                if i >= 95:
                    volume = int(volume * 1.8)

            elif symbol == "BJBY-FX":
                # Steady downtrend with a panic-volume dump, then a late bounce on rumor
                drift = -progress * 35
                price = base + drift + random.uniform(-1.5, 1.5)
                volume = int(1_500_000 + progress * 3_000_000 + random.randint(-200000, 200000))
                if 80 <= i < 85:
                    volume = int(volume * 2.2)
                if i >= 96:
                    price += 6

            else:
                # Range-bound oscillation with a late, directionless volatility spike
                cycle = 8 * random.uniform(0.8, 1.2) * ((i % 20) / 20 - 0.5)
                price = base + cycle + random.uniform(-3, 3)
                volume = int(2_000_000 + random.randint(-300000, 300000))
                if i >= 97:
                    price += random.choice([-9, 9])
                    volume = int(volume * 2)

            execute_db(
                '''INSERT INTO tick (symbol, ts, open, high, low, close, volume)
                   VALUES (:symbol, :ts, :open, :high, :low, :close, :volume)''',
                {
                    "symbol": symbol,
                    "ts": ts,
                    "open": price,
                    "high": price + 2,
                    "low": price - 2,
                    "close": price + random.uniform(-1, 1),
                    "volume": volume
                }
            )

    # Sample sentiment data, aligned with each symbol's story
    print("Loading sample sentiment data...")

    headline_bank = {
        "positive": [
            "Company reports record quarterly earnings",
            "Analyst upgrades stock on strong growth outlook",
            "New product launch exceeds expectations",
            "Institutional investors increase holdings"
        ],
        "negative": [
            "Regulatory investigation raises concerns",
            "Guidance cut spooks investors",
            "Analyst downgrades on weak demand",
            "Executive departure unsettles markets"
        ],
        "neutral": [
            "Market volatility continues amid mixed signals",
            "Trading volume in line with recent average",
            "Sector performance diverges across peers"
        ],
        "rumor": [
            "Unconfirmed report suggests takeover interest"
        ]
    }

    # YWMG-FX gets a deliberately balanced set of headlines, rather than leaving the
    # positive/negative split to chance - that balance is the point of the story.
    ywmg_buckets = ["positive"] * 7 + ["negative"] * 7 + ["neutral"] * 6
    random.shuffle(ywmg_buckets)

    for symbol in symbols:
        for i in range(20):
            ts = now - timedelta(days = random.randint(0, 7), seconds = i * 10)

            if symbol == "BBRQ-FX":
                is_positive = random.random() < 0.9
                headline = random.choice(headline_bank["positive"] if is_positive else headline_bank["neutral"])
                compound = random.uniform(0.6, 0.9) if is_positive else random.uniform(-0.05, 0.05)

            elif symbol == "BJBY-FX":
                if i == 0:
                    headline = headline_bank["rumor"][0]
                    compound = random.uniform(0.5, 0.7)
                else:
                    is_negative = random.random() < 0.9
                    headline = random.choice(headline_bank["negative"] if is_negative else headline_bank["neutral"])
                    compound = random.uniform(-0.9, -0.6) if is_negative else random.uniform(-0.05, 0.05)

            else:
                bucket = ywmg_buckets[i]
                headline = random.choice(headline_bank[bucket])
                compound = {
                    "positive": random.uniform(0.3, 0.5),
                    "negative": random.uniform(-0.5, -0.3),
                    "neutral": random.uniform(-0.05, 0.05)
                }[bucket]

            execute_db(
                '''INSERT INTO stock_sentiment (symbol, ts, headline, compound, publisher)
                   VALUES (:symbol, :ts, :headline, :compound, :publisher)''',
                {
                    "symbol": symbol,
                    "ts": ts,
                    "headline": headline,
                    "compound": compound,
                    "publisher": "sample_data"
                }
            )

    print("Sample data loaded successfully!")

load_sample_data()

Loading sample tick data...
Loading sample sentiment data...
Sample data loaded successfully!


In [20]:
# Analyze a stock (no execution)

context = run_trading_pipeline("BBRQ-FX", execute = False)


[SYSTEM] MULTI-AGENT TRADING SYSTEM: BBRQ-FX

------------------------------------------------------------
[AGENT] DATA_ANALYST
------------------------------------------------------------
Recommendation: BUY
Confidence: 80%
Reasoning: Price has increased significantly by 8.1% in 24h with a high near the current price, indicating strong upward momentum. Sentiment is very positive with a score of 0.69 and no negative mentions, supporting bullish outlook.
Metadata: {'approved': True, 'fallback': False}

------------------------------------------------------------
[AGENT] RISK_MANAGER
------------------------------------------------------------
Recommendation: BUY
Confidence: 80%
Reasoning: Approved with high confidence; confidence level is adequate, recent price momentum and positive sentiment support the recommendation, and no indication of overtrading.
Metadata: {'approved': True, 'fallback': False}

------------------------------------------------------------
[AGENT] PORTFOLIO_OPTIMI

In [21]:
# Analyze and execute trades

context = run_trading_pipeline("BBRQ-FX", execute = True)


[SYSTEM] MULTI-AGENT TRADING SYSTEM: BBRQ-FX

------------------------------------------------------------
[AGENT] DATA_ANALYST
------------------------------------------------------------
Recommendation: BUY
Confidence: 80%
Reasoning: Price increased by 8.1% in 24h with a high near the current price, indicating strong upward momentum. Sentiment is very positive with a score of 0.69 and no negative mentions, supporting bullish outlook.
Metadata: {'approved': True, 'fallback': False}

------------------------------------------------------------
[AGENT] RISK_MANAGER
------------------------------------------------------------
Recommendation: BUY
Confidence: 80%
Reasoning: Confidence level is adequate, recent trading activity supports the recommendation, and frequency of trades is reasonable. Approved with high confidence.
Metadata: {'approved': True, 'fallback': False}

------------------------------------------------------------
[AGENT] PORTFOLIO_OPTIMIZER
-----------------------------

In [22]:
# View decision history

def view_decision_history(symbol: str, limit: int = 10):
    decisions = query_db(
        '''SELECT agent_id, decision_timestamp, action, confidence, reasoning
           FROM agent_decisions
           WHERE symbol = :symbol
           ORDER BY decision_timestamp DESC
           LIMIT :limit''',
        {"symbol": symbol, "limit": limit}
    )

    print(f"\n[HISTORY] Decision History for {symbol}")
    print("=" * 80)
    for d in decisions:
        print(f"{d['decision_timestamp']} | {d['agent_id']:20} | {d['action']:6} | {d['confidence']*100:5.0f}%")
        print(f"  -> {d['reasoning']}\n")

view_decision_history("BBRQ-FX")


[HISTORY] Decision History for BBRQ-FX
2026-07-12 15:03:46 | portfolio_optimizer  | BUY    |    85%
  -> Improves diversification as portfolio is currently empty

2026-07-12 15:03:44 | risk_manager         | BUY    |    80%
  -> Confidence level is adequate, recent trading activity supports the recommendation, and frequency of trades is reasonable. Approved with high confidence.

2026-07-12 15:03:43 | data_analyst         | BUY    |    80%
  -> Price increased by 8.1% in 24h with a high near the current price, indicating strong upward momentum. Sentiment is very positive with a score of 0.69 and no negative mentions, supporting bullish outlook.

2026-07-12 15:03:31 | portfolio_optimizer  | BUY    |    85%
  -> Improves diversification as portfolio is currently empty, and recent momentum and sentiment are positive

2026-07-12 15:03:29 | risk_manager         | BUY    |    80%
  -> Approved with high confidence; confidence level is adequate, recent price momentum and positive sentiment s

In [23]:
# Compare multiple stocks

symbols = ["BBRQ-FX", "BJBY-FX", "YWMG-FX"]
results = {}

for symbol in symbols:
    print(f"\n\n{'#' * 60}")
    print(f"Analyzing {symbol}")
    print(f"{'#' * 60}")
    results[symbol] = run_trading_pipeline(symbol, execute = False)

# Summary
print("\n\n[SUMMARY]")
print("=" * 60)
for symbol, ctx in results.items():
    final = ctx.agent_decisions[-1]
    print(f"{symbol:6} | {final.recommendation:4} | {final.confidence*100:5.0f}% | {final.metadata.get('trade_size', 0):3} shares")



############################################################
Analyzing BBRQ-FX
############################################################

[SYSTEM] MULTI-AGENT TRADING SYSTEM: BBRQ-FX

------------------------------------------------------------
[AGENT] DATA_ANALYST
------------------------------------------------------------
Recommendation: HOLD
Confidence: 70%
Reasoning: Price shows strong 24h gain (+8.1%) with high sentiment score (0.69) and no negative sentiment, but current price is near 24h high indicating potential short-term resistance; holding position to monitor further movement is advisable.
Metadata: {'approved': True, 'fallback': False}

------------------------------------------------------------
[AGENT] RISK_MANAGER
------------------------------------------------------------
Recommendation: BUY
Confidence: 80%
Reasoning: Recent trading activity shows strong buy signals with high confidence levels, and the analyst confidence is above 60%. Although the price is near s